# Task 3: Collaborative Filtering: User-Item Matrix and K-Nearest Neighbours

## a) Build a user-item matrix (utility matrix) where each row represents a user, and each column represents a movie

In [1]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

Load data

In [2]:
ratings = pd.read_csv("ratings.csv")
movies = pd.read_csv("movies.csv")

Generate sparse utility matrix

In [3]:
def create_sparse_utility_matrix(df):
    unique_user_count = df['userId'].nunique()
    unique_movie_count = df['movieId'].nunique()

    user_mapper = dict(zip(np.unique(df["userId"]), list(range(unique_user_count))))
    movie_mapper = dict(zip(np.unique(df["movieId"]), list(range(unique_movie_count))))

    user_inv_mapper = dict(zip(list(range(unique_user_count)), np.unique(df["userId"])))
    movie_inv_mapper = dict(zip(list(range(unique_movie_count)), np.unique(df["movieId"])))

    user_index = [user_mapper[i] for i in df['userId']]
    item_index = [movie_mapper[i] for i in df['movieId']]
    # sparse format: (data, (row_indices, col_indices)) where row_indices are users and col_indices are movies
    sparse_utility_matrix = csr_matrix((df["rating"], (user_index,item_index)), shape=(unique_user_count, unique_movie_count))

    return sparse_utility_matrix, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper

sparse_utility_matrix, user_mapper, movie_mapper, user_inv_mapper, movie_inv_mapper = create_sparse_utility_matrix(ratings)

print("Sparse utility matrix shape:", sparse_utility_matrix.shape)
print("User mapper example:", list(user_mapper.items())[:5])
print("Movie mapper example:", list(movie_mapper.items())[:5])
print("User inverse mapper example:", list(user_inv_mapper.items())[:5])
print("Movie inverse mapper example:", list(movie_inv_mapper.items())[:5])
print("Sparse utility matrix example:\n", sparse_utility_matrix.toarray()[:5, :5])

Sparse utility matrix shape: (610, 9724)
User mapper example: [(1, 0), (2, 1), (3, 2), (4, 3), (5, 4)]
Movie mapper example: [(1, 0), (2, 1), (3, 2), (4, 3), (5, 4)]
User inverse mapper example: [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)]
Movie inverse mapper example: [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5)]
Sparse utility matrix example:
 [[4. 0. 4. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]
 [4. 0. 0. 0. 0.]]


## b) Calculate the sparsity of the user-item matrix and explain the results

In [4]:
n_total = sparse_utility_matrix.shape[0]*sparse_utility_matrix.shape[1] # total number of entries where shape[0] is number of users and shape[1] is number of movies
n_ratings = sparse_utility_matrix.nnz # number of non-zero entries
sparsity = n_ratings/n_total # sparsity ratio
print(f"Matrix sparsity: {round(sparsity*100,2)}%")

if sparsity < 0.05:
    print("The matrix is sparse (less than 5% filled).")
    print("This indicates that most users have rated only a small fraction of the available movies.")
    print("With so few ratings, it may be difficult for recommendation algorithms to find patterns and make accurate predictions.")
    print("New users or items will struggle to get meaningful recommendations due to insufficient data.")
else:
    print("The matrix is not sparse (greater than 5% filled).")
    print("This indicates that users have rated a larger portion of the available movies.")
    print("With more ratings, recommendation algorithms can better identify patterns and make accurate predictions.")
    print("New users or items will have a better chance of receiving meaningful recommendations due to more available data.")

Matrix sparsity: 1.7%
The matrix is sparse (less than 5% filled).
This indicates that most users have rated only a small fraction of the available movies.
With so few ratings, it may be difficult for recommendation algorithms to find patterns and make accurate predictions.
New users or items will struggle to get meaningful recommendations due to insufficient data.


## c) Use the k-nearest neighbours algorithm to recommend movies based on user ratings. Implement a function to recommend the top 5 movies for a given user based on their rating history

Function to recommend movies to a user by using kNN

In [5]:
def recommend_movies_knn(user_id, utility_matrix, user_mapper, movie_inv_mapper, movies_df, k=5, top_n=5):
    # Map user_id to matrix index
    user_idx = user_mapper.get(user_id)
    if user_idx is None:
        return pd.DataFrame([{"error": f"User {user_id} not found in dataset"}])

    # Extract user's ratings row
    user_vector = utility_matrix[user_idx]
    if hasattr(user_vector, 'toarray'):
        user_vector = user_vector.toarray().reshape(1, -1)
    else:
        user_vector = np.asarray(user_vector).reshape(1, -1)

    # Fit NearestNeighbors, k+1 to include self
    knn = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute')
    knn.fit(utility_matrix)

    # Find k nearest neighbors (excluding self)
    distances, neighbor_indices = knn.kneighbors(user_vector)
    neighbor_indices, distances = neighbor_indices[0][1:], distances[0][1:]

    # Identify movies already rated by user
    user_ratings = utility_matrix[user_idx]
    if hasattr(user_ratings, 'toarray'):
        user_ratings = user_ratings.toarray().flatten()
    else:
        user_ratings = np.asarray(user_ratings).flatten()

    user_rated_movies = {
        movie_inv_mapper[i]
        for i, rating in enumerate(user_ratings) if rating > 0
    }

    # Predict scores for unrated movies using weighted average from neighbors
    movie_scores = {}
    for movie_idx in range(utility_matrix.shape[1]):
        movie_id = movie_inv_mapper[movie_idx]
        if movie_id in user_rated_movies: # skip already rated movies
            continue

        ratings = []
        similarity_sum = []
        for i, neighbor_idx in enumerate(neighbor_indices):
            neighbor_rating = utility_matrix[neighbor_idx, movie_idx]
            if hasattr(neighbor_rating, 'toarray'):
                neighbor_rating = neighbor_rating.toarray()[0, 0]
            elif hasattr(neighbor_rating, 'item'):
                neighbor_rating = neighbor_rating.item()

            if neighbor_rating > 0:
                sim = 1 - distances[i] # convert distance to similarity, use 1 - distance
                ratings.append(neighbor_rating * sim) # weighted rating, use rating * similarity
                similarity_sum.append(sim)

        if similarity_sum:
            movie_scores[movie_id] = sum(ratings) / sum(similarity_sum) # weighted average, use sum(rating * similarity) / sum(similarity)

    # Sort and select top N for not rated movies
    top_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    # Build recommendations DataFrame
    recommendations = [
        {
            "movieId": mid,
            "title": movies_df.loc[movies_df['movieId'] == mid, 'title'].values[0],
            "predicted_rating": round(score, 2)
        }
        for mid, score in top_movies
        if not movies_df.loc[movies_df['movieId'] == mid].empty
    ]

    return pd.DataFrame(recommendations)


Retrieve top 5 similar movies for a given user

In [6]:
user_id = 5

print(f"Recommending movies for User {user_id}:")
print("=" * 50)

recommendations = recommend_movies_knn(
    user_id=user_id,
    utility_matrix=sparse_utility_matrix,
    user_mapper=user_mapper,
    movie_inv_mapper=movie_inv_mapper,
    movies_df=movies,
    k=5,
    top_n=5
)

print("Top 5 Movie Recommendations:")
recommendations

Recommending movies for User 5:
Top 5 Movie Recommendations:


,movieId,title,predicted_rating
0,25,Leaving Las Vegas (1995),5.00
1,236,French Kiss (1995),5.00
2,780,Independence Day (a.k.a. ID4) (1996),5.00
3,593,"Silence of the Lambs, The (1991)",4.65
4,356,Forrest Gump (1994),4.41


## d) Handle the cold-start problem for new users by using item-based collaborative filtering to recommend movies similar to those the user has already rated

The cold-start problem occurs when:
- New users join the system with no rating history
- New items are added with no ratings from existing users
- The system cannot find similar users/items to make recommendations

Item-Based for Cold-Start
- Even new users typically rate a few movies initially
- We can find movies similar to these initial ratings
- Item similarities are more stable than user similarities
- Works well with minimal user data

Function to recommend movies to a new user

In [7]:
def recommend_for_new_user_item_based(utility_matrix, user_rated_movies, user_ratings_dict, movie_inv_mapper, movies_df, top_n=5):
    # Compute item-item similarity (works with sparse matrix)
    item_similarity_matrix = cosine_similarity(utility_matrix.T)

    # Precompute reverse mapping: movieId -> index
    movie_to_idx = {movie_id: idx for idx, movie_id in movie_inv_mapper.items()}

    # Filter rated movie indices only to those present in the mapping
    rated_indices = [movie_to_idx[m] for m in user_rated_movies if m in movie_to_idx]

    movie_scores = {}

    for movie_idx, movie_id in movie_inv_mapper.items():
        # Skip already-rated movies
        if movie_id in user_rated_movies:
            continue

        weighted_sum = 0
        sim_sum = 0

        for rated_idx in rated_indices:
            similarity = item_similarity_matrix[movie_idx, rated_idx]
            user_rating = user_ratings_dict.get(movie_inv_mapper[rated_idx], 0)

            if user_rating > 0:
                weighted_sum += similarity * user_rating
                sim_sum += abs(similarity)

        if sim_sum > 0:
            movie_scores[movie_id] = weighted_sum / sim_sum

    # Select top-N from not rated movies
    top_movies = sorted(movie_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    # Build recommendations DataFrame
    recommendations = [
        {
            "movieId": mid,
            "title": movies_df.loc[movies_df['movieId'] == mid, 'title'].values[0],
            "predicted_rating": round(score, 2)
        }
        for mid, score in top_movies
        if not movies_df.loc[movies_df['movieId'] == mid].empty
    ]

    return pd.DataFrame(recommendations)

In [8]:
print("=" * 60)
print("Cold-Start Problem Solution: Item-Based Collaborative Filtering")
print("=" * 60)

# Simulate a new user who has rated a few movies
new_user_ratings = {
    1: 5.0,      # Toy Story (1995) - loved it
    260: 4.5,    # Star Wars (1977) - really liked it
    1196: 4.0,   # Star Wars: Episode V (1980) - liked it
    2628: 3.5    # Star Wars: Episode I (1999) - okay
}

print("New User's Ratings:")
print("-" * 30)
for movie_id, rating in new_user_ratings.items():
    movie_info = movies[movies['movieId'] == movie_id]
    if not movie_info.empty:
        title = movie_info.iloc[0]['title']
        print(f"• {title}: {rating}/5.0")

print(f"\nGenerating recommendations based on item similarity...")
print("-" * 50)

# Get recommendations using item-based collaborative filtering
recommendations = recommend_for_new_user_item_based(
    utility_matrix=sparse_utility_matrix,
    user_rated_movies=list(new_user_ratings.keys()),
    user_ratings_dict=new_user_ratings,
    movie_inv_mapper=movie_inv_mapper,
    movies_df=movies,
    top_n=5
)

print("\nTop 5 Recommendations for New User:")
recommendations


Cold-Start Problem Solution: Item-Based Collaborative Filtering
New User's Ratings:
------------------------------
• Toy Story (1995): 5.0/5.0
• Star Wars: Episode IV - A New Hope (1977): 4.5/5.0
• Star Wars: Episode V - The Empire Strikes Back (1980): 4.0/5.0
• Star Wars: Episode I - The Phantom Menace (1999): 3.5/5.0

Generating recommendations based on item similarity...
--------------------------------------------------

Top 5 Recommendations for New User:


,movieId,title,predicted_rating
0,40,"Cry, the Beloved Country (1995)",5.0
1,83,Once Upon a Time... When We Were Colored (1995),5.0
2,117,"Young Poisoner's Handbook, The (1995)",5.0
3,137,Man of the Year (1995),5.0
4,148,"Awfully Big Adventure, An (1995)",5.0
